In [ ]:
import librosa 
from IPython.display import Audio
from tqdm import tqdm
import yaml
from yaml.loader import FullLoader
import os
from os.path import join
import pandas as pd

In [ ]:
def get_yaml_data(path):
    with open(path) as f:
        data = yaml.load(f, Loader=FullLoader)
        return data                            
    
def get_text_data(path):
    with open(path, encoding='utf-8-sig', buffering=1) as f:
        data = tuple(map(str.strip, f.readlines()))
        return data   

In [ ]:
DataPath   = '/kaggle/input/must-c-en-ar/'
data_split = os.listdir(DataPath)
data_split.remove('docs')
print(data_split)

In [ ]:
def create_split_data(split_name):
    split_name  = split_name.strip()
    assert split_name in data_split, f"{split_name} folder dosen't exist on this dataset, only {', '.join(data_split)} are exist"
    data = dict()
    split_path         = join(DataPath, split_name)
    
    if split_name == 'train':
        
        data['en_text']    = get_text_data(join(split_path, 'txt', f'{split_name}_en.txt'))
        data['ar_text']    = get_text_data(join(split_path, 'txt', f'{split_name}_ar.txt'))
    else:
        data['en_text']    = get_text_data(join(split_path, 'txt', f'{split_name}.en'))
        data['ar_text']    = get_text_data(join(split_path, 'txt', f'{split_name}.ar'))
        
    yaml_data          = get_yaml_data(join(split_path, 'txt', f'{split_name}.yaml'))
    data['duration']   = [idx['duration'] for idx in tqdm(yaml_data, unit='row',   desc='   durations',)]
    data['offset']     = [idx['offset'] for idx in tqdm(yaml_data, unit='row',     desc='   offsets  ',)]
    data['speaker_id'] = [idx['speaker_id'] for idx in tqdm(yaml_data, unit='row', desc='   speakers ',)]
    data['wav']        = [idx['wav'] for idx in tqdm(yaml_data, unit='row',        desc='   wavs     ',)]; del yaml_data
    data = pd.DataFrame(data)
    data = data.sample(frac=1).reset_index(drop=True)                     # shuffle data before return it
    return data

In [ ]:
print('dev set')
dev = create_split_data('dev')

print('\ntrain set')
train = create_split_data('train')

print('\ntst_COMMON set')
tst_COMMON = create_split_data('tst-COMMON')

print('\ntst-HE set')
tst_HE = create_split_data('tst-HE')

# helper 2

In [ ]:
from tokenizers import Tokenizer
from tokenizers import decoders

In [ ]:
class TokenHandler:
    def __init__(self, json_path: str):
        self.tok = Tokenizer.from_file(json_path)
        self.tok.enable_padding(pad_id=self.get_id("<PAD>"), pad_token="<PAD>")
        
    def enocde_line(self, text: str):
        '''@input: text --> single string.
        @return:  ids, tokens
        '''
        out = self.tok.encode(text)
        return out.ids, out.tokens
    
    def get_id(self, token: int):
        '''@input: token --> single word 
        @return: id --> int
        '''
        return self.tok.token_to_id(token)
    
    def encode_batch(self, data: list):
        '''@input: data --> list of strings.
        @return:  ids, tokens
        '''
        output = self.tok.encode_batch(data)
        return [o.ids for o in output], [o.tokens for o in output]
    
    def decode_line(self, ids: list):
        '''@input: ids --> list of int
        @return: text --> single string.
        '''
        return self.tok.decode(ids)
    
    def decode_batch(self, ids: list):
        return self.tok.decode_batch(ids)

In [ ]:
# path = r"/kaggle/working/tokens/libri_tokenizer.json"
# en_token = TokenHandler(path)
# text = "hello, how're you? I hope you are fine."
# ids = en_token.enocde_line(text)
# print(ids)
# # tok = en_token.decode_batch(ids)
# # print(tok)